In [ ]:
import pandas as pd

In [11]:
path = "/Users/philipp/Downloads/Iran/iranian_tweets_control.pkl.gz"
df_control = pd.read_pickle(path)

sample = df.sample(20).copy()
sample.to_clipboard(sep="\t", index=False)

In [6]:
path = "/Users/philipp/Downloads/Iran/iranian_tweets_io.pkl.gz"
df_io = pd.read_pickle(path)

sample = df_io.sample(20).copy()
sample.to_clipboard(sep="\t", index=False)

In [25]:
import pandas as pd
import networkx as nx

# 1. Pull just the retweet edges from each df, tag the source dataset
cols = ['userid', 'retweet_userid', 'retweet_tweetid', 'tweet_time']

def edges(df, label):
    e = df.loc[df['is_retweet'] == True, cols].copy()
    e = e.dropna(subset=['userid', 'retweet_userid'])
    # avoid self-loops (some dumps have them)
    e = e[e['userid'] != e['retweet_userid']]
    e['dataset'] = label
    return e

edges_all = pd.concat([edges(df_io, 'io'),
                       edges(df_control, 'control')],
                      ignore_index=True)

# 2. Aggregate to weighted edges
#    weight = total retweets, plus per-dataset counts so you can color/filter later
agg = (edges_all
       .groupby(['userid', 'retweet_userid', 'dataset'])
       .size().unstack(fill_value=0)
       .reset_index()
       .rename(columns={'io': 'w_io', 'control': 'w_control'}))
agg['weight'] = agg.get('w_io', 0) + agg.get('w_control', 0)

# 3. Build the graph (edge: retweeter -> original author)
G = nx.from_pandas_edgelist(
    agg,
    source='userid',
    target='retweet_userid',
    edge_attr=['weight', 'w_io', 'w_control'],
    create_using=nx.DiGraph,
)

# 4. Tag each node by which dataset(s) it appears in
io_users      = set(df_io['userid']).union(df_io['retweet_userid'].dropna())
control_users = set(df_control['userid']).union(df_control['retweet_userid'].dropna())
for n in G.nodes:
    in_io, in_ctrl = n in io_users, n in control_users
    G.nodes[n]['group'] = ('both' if in_io and in_ctrl
                           else 'io' if in_io else 'control')

print(G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')

71542 nodes, 118780 edges


In [26]:
nx.write_graphml(G, '/Users/philipp/Downloads/Iran/io_hunter_iran.graphml')